In [1]:
import pandas as pd
import zipfile 
zip_path = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\price data\csv.zip"
with zipfile.ZipFile(zip_path,'r') as z:
    first_csv= [f for f in z.namelist() if f.endswith('.csv')][0]
    with z.open(first_csv) as f:
        columns= pd.read_csv(f, nrows=0).columns.tolist()
for col in columns:
    print(f"{col}")
        

C:\Users\Kanishk Rathore\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


State
District
Market
Commodity
Variety
Grade
Arrival_Date
Min_Price
Max_Price
Modal_Price
Commodity_Code


In [1]:
#extracting the time series of tomato prices without loading the data
import pandas as pd
import zipfile
zip_path = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\price data\csv.zip"

data_pieces = []

with zipfile.ZipFile(zip_path, 'r') as z:
    for year in range(2015, 2026): 
        file_name = f"{year}.csv"
        if file_name in z.namelist():
            with z.open(file_name) as f:
                df_year= pd.read_csv(f,usecols=['State', 'Commodity', 'Arrival_Date', 'Modal_Price', 'Variety'])
                data_pieces.append(df_year)                    
                                 
df = pd.concat(data_pieces, ignore_index=True)

df['State'] = df['State'].str.title().str.strip()
df['Commodity'] = df['Commodity'].str.title().str.strip()
df['Variety']= df['Variety'].str.title().str.strip()


is_ap = df['State'] == 'Andhra Pradesh'
is_tomato = df['Commodity'] == 'Tomato'
is_variety= df['Variety']== 'Local'

df_ap = df[is_ap & is_tomato & is_variety].copy()

df_ap['Arrival_Date'] = pd.to_datetime(df_ap['Arrival_Date'])


df_daily = df_ap.groupby('Arrival_Date')['Modal_Price'].mean().reset_index()
df_daily = df_daily.sort_values('Arrival_Date')
df_daily = df_daily.set_index('Arrival_Date')
df_daily = df_daily.asfreq('D')
df_daily['Modal_Price'] = df_daily['Modal_Price'].ffill(limit=1)
df_daily['Modal_Price'] = df_daily['Modal_Price'].interpolate(method='linear')

print(df_daily.head(10))
print(len(df_daily))

C:\Users\Kanishk Rathore\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


              Modal_Price
Arrival_Date             
2015-01-01     660.000000
2015-01-02     700.000000
2015-01-03     950.000000
2015-01-04     700.000000
2015-01-05     714.285714
2015-01-06     700.000000
2015-01-07     728.571429
2015-01-08     642.857143
2015-01-09     621.428571
2015-01-10     564.285714
4016


In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import zipfile
from statsmodels.tsa.stattools import adfuller, kpss
import scipy.optimize as optimize
import warnings

# 1. Extract and build the daily tomato price series (Step 1)
zip_path = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\price data\csv.zip"
data_pieces = []

with zipfile.ZipFile(zip_path, 'r') as z:
    for year in range(2015, 2026): 
        file_name = f"{year}.csv"
        if file_name in z.namelist():
            with z.open(file_name) as f:
                df_year = pd.read_csv(f, usecols=['State', 'Commodity', 'Arrival_Date', 'Modal_Price', 'Variety'])
                data_pieces.append(df_year)                            
                             
df = pd.concat(data_pieces, ignore_index=True)

df['State'] = df['State'].str.title().str.strip()
df['Commodity'] = df['Commodity'].str.title().str.strip()
df['Variety'] = df['Variety'].str.title().str.strip()

is_ap = df['State'] == 'Andhra Pradesh'
is_tomato = df['Commodity'] == 'Tomato'
is_variety = df['Variety'] == 'Local'

df_ap = df[is_ap & is_tomato & is_variety].copy()
df_ap['Arrival_Date'] = pd.to_datetime(df_ap['Arrival_Date'])

df_daily = df_ap.groupby('Arrival_Date')['Modal_Price'].mean().reset_index()
df_daily = df_daily.sort_values('Arrival_Date')
df_daily = df_daily.set_index('Arrival_Date')
df_daily = df_daily.asfreq('D')
df_daily['Modal_Price'] = df_daily['Modal_Price'].ffill(limit=1)
df_daily['Modal_Price'] = df_daily['Modal_Price'].interpolate(method='linear')

# 2. Deflate using the monthly price index (Base 100)
index_path = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\tomato\price_index.xlsx"
df_index = pd.read_excel(index_path)
df_index.columns = df_index.columns.str.strip()

clean_str = df_index['Month'].str.replace('INDX', '', regex=False)
df_index['Year'] = clean_str.str[2:].astype(int)
df_index['Month_Num'] = clean_str.str[:2].astype(int)
df_index['YearMonth'] = df_index['Year'].astype(str) + '-' + df_index['Month_Num'].astype(str).str.zfill(2)

df_daily = df_daily.reset_index()
df_daily['Arrival_Date'] = pd.to_datetime(df_daily['Arrival_Date'])
df_daily['YearMonth'] = df_daily['Arrival_Date'].dt.to_period('M').astype(str)

df_daily = df_daily.merge(
    df_index[['YearMonth', 'Price Index']], 
    on='YearMonth', 
    how='left'
)

df_daily['Real_Modal_Price'] = (df_daily['Modal_Price'] / df_daily['Price Index']) * 100
df_daily = df_daily.set_index('Arrival_Date')
df_daily = df_daily.drop(columns=['YearMonth'])

# 3. Take Natural Log & Run OLS Regression for Deseasonalization (Steps 2 & 3)
df_daily['Log_Real_Modal_Price'] = np.log(df_daily['Real_Modal_Price'])

df_daily['Month'] = df_daily.index.month
df_daily['DOW'] = df_daily.index.dayofweek

month_dummies = pd.get_dummies(df_daily['Month'], prefix='Month', drop_first=True, dtype=int)
dow_dummies = pd.get_dummies(df_daily['DOW'], prefix='DOW', drop_first=True, dtype=int)

X = pd.concat([month_dummies, dow_dummies], axis=1)
X = sm.add_constant(X) 
y = df_daily['Log_Real_Modal_Price']

ols_model = sm.OLS(y, X).fit()
df_daily['Deseasonalized_Log_Price'] = ols_model.resid

# 4. Statistical Testing on Residuals ($\tilde{p}_t$) (Step 4)
series = df_daily['Deseasonalized_Log_Price'].dropna()

print("=" * 50)
print("AUGMENTED DICKEY-FULLER (ADF) TEST")
print("=" * 50)
adf_aic = adfuller(series, autolag='AIC')
adf_bic = adfuller(series, autolag='BIC')

print(f"ADF (AIC Lags) Stat: {adf_aic[0]:.4f} | p-value: {adf_aic[1]:.4f} | Lags: {adf_aic[2]}")
print(f"ADF (BIC Lags) Stat: {adf_bic[0]:.4f} | p-value: {adf_bic[1]:.4f} | Lags: {adf_bic[2]}")

print("\n" + "=" * 50)
print("KPSS TEST")
print("=" * 50)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    kpss_res = kpss(series, regression='c', nlags='auto')
print(f"KPSS Stat: {kpss_res[0]:.4f} | p-value: {kpss_res[1]:.4f}")

print("\n" + "=" * 50)
print("LOCAL WHITTLE ESTIMATOR")
print("=" * 50)
def local_whittle(x, m=None):
    x = np.asarray(x)
    n = len(x)
    if m is None:
        m = int(n**0.5)  # Standard bandwidth choice
    
    j = np.arange(1, m + 1)
    fft_x = np.fft.fft(x - np.mean(x))
    I_omega = (np.abs(fft_x[1:m+1]) ** 2) / (2 * np.pi * n)
    
    def objective(d):
        return np.log(np.mean(I_omega * (j ** (2 * d)))) - 2 * d * np.mean(np.log(j))
    
    res = optimize.minimize_scalar(objective, bounds=(-0.5, 1.5), method='bounded')
    return res.x

d_lw = local_whittle(series)
print(f"Estimated Fractional Integration Parameter (d): {d_lw:.4f}")
print("=" * 50)

AUGMENTED DICKEY-FULLER (ADF) TEST
ADF (AIC Lags) Stat: -6.6835 | p-value: 0.0000 | Lags: 31
ADF (BIC Lags) Stat: -8.8981 | p-value: 0.0000 | Lags: 3

KPSS TEST
KPSS Stat: 0.5614 | p-value: 0.0278

LOCAL WHITTLE ESTIMATOR
Estimated Fractional Integration Parameter (d): 0.3323


C:\Users\Kanishk Rathore\AppData\Local\Temp\ipykernel_18604\2836726716.py:87: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_aic = adfuller(series, autolag='AIC')
C:\Users\Kanishk Rathore\AppData\Local\Temp\ipykernel_18604\2836726716.py:88: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_bic = adfuller(series, autolag='BIC')
